# PTM3D Polar Survey Pipeline

Interactive notebook for running end-to-end illumination and thermal surveys of lunar polar regions.

**Workflow:**
1. Configure ROI, time period, and selection criteria
2. Build mesh from DEM
3. Run illumination survey (25 lunations)
4. Compute illumination duration maps
5. Select target sites
6. Run thermal simulations (optional)
7. Plan traverse route (optional)
8. Generate maps and report

Edit the configuration in Cell 2 to run different cases.

In [ ]:
# ========================================================================
# SETUP — run this cell first (only once per session)
# ========================================================================
import sys, os
from pathlib import Path

# Project root
PROJECT = Path("/Users/paha3326/Library/CloudStorage/OneDrive-UCB-O365/europa/heat3d/ptm")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts" / "survey"))

from survey_config import SurveyConfig, load_config
from utils import read_ply_mesh, read_stats_binary, haversine_distance
import numpy as np

print(f"Working directory: {PROJECT}")
print("Survey pipeline modules loaded.")

## 1. Configuration

Edit the parameters below to define your survey. All settings are in one place.

In [ ]:
# ========================================================================
# CONFIGURATION — edit these parameters for each survey
# ========================================================================
import yaml, tempfile

config = {
    # --- Survey identity ---
    "survey": {
        "name": "amundsen_2026",
        "body": "Moon",
        "body_radius": 1737400.0,
    },

    # --- Region of interest ---
    "roi": {
        "center_lat": -84.5,         # crater center latitude
        "center_lon": 85.5,          # crater center longitude (East)
        "radius_km": 75,             # extraction radius
        "max_points": 2000000,       # max mesh vertices
        "density_exp": 1.0,          # density falloff (1=linear, 2=quadratic)
        "core_frac": 0.10,           # fraction at full resolution
    },

    # --- DEM source ---
    "dem": {
        "file": "/Users/paha3326/Library/CloudStorage/OneDrive-UCB-O365/moon/lola/dem/LDEM_80S_20M.LBL",
        "south_pole_coords": True,
    },

    # --- Time period ---
    "time": {
        "start": "2026-04-01T00:00:00",
        "duration_years": 2.0,
        "lunation_days": 29.530588,
        "timesteps_per_lunation": 360,   # ~2 hr resolution
    },

    # --- Illumination survey ---
    "illumination": {
        "num_lunations": 25,
        "compute_duration_maps": True,
    },

    # --- Thermal model ---
    "thermal": {
        "enabled": True,
        "lunations": [7, 13],           # summer + winter
        "equilibration_orbits": 1.0,
        "radiation_exchange": True,
        "max_coupling_iterations": 1,
        "use_gpu": True,
    },

    # --- Site selection criteria ---
    "selection_criteria": {
        "min_illum_fraction": 0.0001,    # exclude PSR
        "max_illum_fraction": 0.20,      # exclude always-lit
        "min_continuous_hours": 8.0,     # minimum observation window
        "max_temperature_K": 150.0,      # volatile stability
        "min_cluster_size": 50,
    },

    # --- Named sites (always included) ---
    "named_sites": [
        {"name": "Wilcoski", "lat": -83.93341, "lon": 92.51474},
    ],

    # --- Traverse planning ---
    "traverse": {
        "enabled": True,
        "start_lat": -84.5,
        "start_lon": 85.5,
        "start_time": "2026-10-01T00:00:00",
        "max_distance_km": 50.0,
        "max_duration_days": 14.0,
        "return_to_start": True,
        "min_sites": 3,
        "max_slope_deg": 15.0,
        "min_rest_hours": 2.0,
        "traverse_speed_kmh": 2.0,
        "require_illuminated": True,
        "viewshed": {
            "enabled": False,
            "observer_height_m": 2.0,
            "max_range_km": 10.0,
        },
    },

    # --- Output ---
    "output": {
        "base_dir": "output/amundsen_2026",
        "report_format": "html",
        "clean_per_timestep_data": True,
    },
}

# Write config to temp file and load as SurveyConfig
config_path = Path(config["output"]["base_dir"]) / "survey_config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

cfg = load_config(str(config_path))
errors = cfg.validate()
if errors:
    print("⚠ Configuration errors:")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"✓ Configuration valid: {cfg.name}")
    print(f"  Body: {cfg.body}, ROI: ({cfg.roi.center_lat}°, {cfg.roi.center_lon}°E), r={cfg.roi.radius_km} km")
    print(f"  Time: {cfg.time.start}, {cfg.illumination.num_lunations} lunations")
    print(f"  Output: {cfg.output.base_dir}")

## 2. Build Mesh

Generate a triangulated PLY mesh from the DEM, centered on the ROI. Skips if mesh already exists.

In [ ]:
from build_mesh import build_mesh

if cfg.mesh_path.exists():
    print(f"Mesh already exists: {cfg.mesh_path}")
    print(f"  Delete it to regenerate: rm {cfg.mesh_path}")
else:
    build_mesh(cfg)

## 3. Illumination Survey

Run PTM3D illumination model for each lunation. Then compute per-face duration maps (% time illuminated, longest continuous window). This is the most time-consuming step (~6 hours for 25 lunations with 4M faces).

In [ ]:
from run_illumination import run_illumination_survey, compute_duration_maps

# Check if already done
frac_file = cfg.survey_dir / "illum_total_fraction.bin"
if frac_file.exists():
    print(f"Duration maps already exist. Delete to recompute:")
    print(f"  rm {frac_file}")
    print(f"  rm {cfg.survey_dir / 'illum_max_continuous_hours.bin'}")
else:
    run_illumination_survey(cfg)
    if cfg.illumination.compute_duration_maps:
        compute_duration_maps(cfg)

## 4. Site Selection

Apply selection criteria to identify target sites. Adjust thresholds in the config cell above and re-run this cell to explore different selections.

In [ ]:
from select_sites import select_sites

# Re-read config in case thresholds were changed above
cfg = load_config(str(config_path))
select_sites(cfg)

## 5. Thermal Simulations (Optional)

Run thermal + radiation exchange for summer and winter lunations. Takes several hours per lunation (view factors computed once, then cached).

In [ ]:
from run_thermal import run_thermal

if cfg.thermal.enabled:
    run_thermal(cfg)
else:
    print("Thermal simulations disabled. Set thermal.enabled: true in config to run.")

## 6. Traverse Planning (Optional)

Plan a rover/astronaut traverse visiting target sites, subject to distance, duration, slope, and illumination constraints.

In [ ]:
from plan_traverse import plan_traverse

if cfg.traverse.enabled:
    plan_traverse(cfg)
else:
    print("Traverse planning disabled. Set traverse.enabled: true in config.")

## 7. Maps and Visualization

Generate polar stereographic maps of classification, illumination, temperature, and the seasonal pattern.

In [ ]:
from plot_maps import plot_all
from IPython.display import display, Image as IPImage

plot_all(cfg)

# Display generated figures inline
for fig_name in ['classification.png', 'illum_fraction.png', 'continuous_duration.png',
                 'seasonal_pattern.png', 'temp_summer_avg.png', 'temp_winter_avg.png',
                 'temp_summer_max.png', 'temp_winter_max.png']:
    fig_path = cfg.figures_dir / fig_name
    if fig_path.exists():
        print(f"\n{fig_name}:")
        display(IPImage(filename=str(fig_path), width=700))

## 8. Generate HTML Report

Create a standalone HTML report with all figures, tables, and analysis embedded.

In [ ]:
from generate_report import generate_report

generate_report(cfg, compact=False)
generate_report(cfg, compact=True)

report_path = Path(cfg.output.base_dir) / "report.html"
print(f"\nOpen in browser: file://{report_path.resolve()}")

## 9. Pipeline Status

Check which steps have been completed.

In [ ]:
from run_survey import print_status
print_status(cfg)

---

## Quick-Start: Running a Different Case

To survey a different crater, edit the configuration cell (Cell 2):

1. Change `roi.center_lat` and `roi.center_lon` to the new crater center
2. Adjust `roi.radius_km` for the crater size
3. Update `output.base_dir` to a new directory
4. Optionally add named sites of interest
5. Run cells 2 through 8 in order

**Examples:**

| Crater | Lat | Lon (°E) | Radius (km) |
|--------|-----|----------|-------------|
| Amundsen | -84.5 | 85.5 | 75 |
| Shoemaker | -88.1 | 44.9 | 60 |
| Faustini | -87.3 | 77.0 | 50 |
| Cabeus | -85.3 | -35.7 | 55 |
| Nobile | -85.2 | 53.3 | 45 |